# Treating Unstructured Data Like a DataFrame: An Introduction to DataChain

## Setup

In [ ]:
%pip install --upgrade -q  ipywidgets "datachain[torch]" transformers hf_transfer jiwer huggingface_hub

In [ ]:
import json

import datachain as dc
import jiwer
import numpy as np
import torch
from datachain import C, DataModel, File
from datachain.func import path
from datachain.lib.hf import HFAudio
from google.colab import userdata
from huggingface_hub import hf_hub_download, login
from IPython.display import Audio
from transformers import Pipeline, pipeline
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

# Set up device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

# Authenticate with Hugging Face
HF_KEY = userdata.get('HF_KEY')
login(token=HF_KEY)

## Loading data

In [ ]:
dc.read_storage("hf://datasets/MLCommons/peoples_speech/").show(15)

Preparing: 0 rows [00:00, ? rows/s]

Listing hf://datasets: 0 objects [00:00, ? objects/s]

Generated: 0 rows [00:00, ? rows/s]

Processed: 0 rows [00:00, ? rows/s]

,file,file
,path,size
0,MLCommons/peoples_speech/.gitattributes,2168
1,MLCommons/peoples_speech/.gitignore,11
2,MLCommons/peoples_speech/README.md,11530
3,MLCommons/peoples_speech/n_shards.json,212
4,MLCommons/peoples_speech/microset/train-00000-...,90204303
5,MLCommons/peoples_speech/validation/validation...,467578070
6,MLCommons/peoples_speech/validation/validation...,529766184
7,MLCommons/peoples_speech/validation/validation...,367066121
8,MLCommons/peoples_speech/validation/validation...,503637714



[Limited by 15 rows]


In [ ]:
dc.read_storage("hf://datasets/MLCommons/peoples_speech/microset").show(10)

,file,file
,path,size
0,MLCommons/peoples_speech/microset/train-00000-...,90204303


In [ ]:
audio_chain = (
    dc.read_hf("MLCommons/peoples_speech", "microset", streaming=True, limit=20)
    .filter(C("duration_ms") > 14800)
    # .settings(cache=True)
    # .persist()
)
audio_chain.show()

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/804 [00:00<?, ?it/s]

Preparing: 0 rows [00:00, ? rows/s]

Generated: 0 rows [00:00, ? rows/s]

Processed: 0 rows [00:00, ? rows/s]

...

Resolving data files:   0%|          | 0/804 [00:00<?, ?it/s]

Parsed Hugging Face dataset split 'train': 0 rows [00:00, ? rows/s]

Generated: 0 rows [00:00, ? rows/s]

Processed: 0 rows [00:00, ? rows/s]

,id,audio,audio,duration_ms,text
,,array,sampling_rate,,
0,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,"[0.142059326171875, 0.206207275390625, 0.27151...",16000,14920,i wanted this to share a few things but i'm go...
1,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,"[-0.00714111328125, -0.01025390625, -0.0123596...",16000,14830,three and we produce last year we produce twen...
2,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,"[-0.005462646484375, 0.013641357421875, 0.0326...",16000,14940,commons here in the spirit of being able to gr...
3,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,"[-0.0050048828125, -0.00830078125, -0.01501464...",16000,14920,farmers but gardeners and foodies alike who va...
4,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,"[-0.059356689453125, -0.11090087890625, -0.158...",16000,14970,at forty years old getting into farming didn't...
5,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,"[0.010406494140625, 0.004302978515625, -0.0062...",16000,14830,daughter who came out of it actually she desig...
6,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,"[0.021209716796875, -0.008056640625, -0.033142...",16000,14980,has her own business in that respect so there'...
7,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,"[-0.052734375, -0.0084228515625, 0.01745605468...",16000,14820,advocating advocating and working on developin...
8,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,"[-0.00433349609375, -0.00201416015625, -0.0016...",16000,14860,five point plan and you the folks that are run...


In [ ]:
# 1. Grab the first 'audio' object from the list
audio_data = audio_chain.limit(2).to_values("audio")[1]

# 2. Use dot notation (.array) instead of dictionary keys (["array"])
Audio(data=audio_data.array, rate=audio_data.sampling_rate)

UDF 'read_values': Skipped, reusing output from checkpoint
UDF '<unknown>': Skipped, reusing output from checkpoint


## Generating transcript v1

In [ ]:
# Swap File for HFAudio and change the function name
def generate_transcript(helper: Pipeline, audio: HFAudio) -> str:
    audio_array = np.array(audio.array, dtype=np.float32)
    result = helper(audio_array)

    return result["text"]

# Apply the mapping to the dataset
text_chain = (
    audio_chain
    .setup(helper=lambda: pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-tiny",
        dtype=torch.float32,
        device=device,
        generate_kwargs={"language": "en", "task": "transcribe"},
    ))
    .map(generated_text=generate_transcript)
    .persist()
)

# Show the results (comparing the original 'text' to the 'generated_text')
text_chain.select("id", "text", "generated_text").show()

UDF 'read_values': Skipped, reusing output from checkpoint
UDF '<unknown>': Skipped, reusing output from checkpoint


Preparing: 0 rows [00:00, ? rows/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

Processed: 0 rows [00:00, ? rows/s]

,id,text,generated_text
0,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,i wanted this to share a few things but i'm go...,"I wanted to share a few things, but I'm not g..."
1,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,three and we produce last year we produce twen...,"And we produced last year we produced 21,000 ..."
2,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,commons here in the spirit of being able to gr...,comments here in the spirit of being able to ...
3,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,farmers but gardeners and foodies alike who va...,"farmers, but gardeners and foodies alike who ..."
4,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,at forty years old getting into farming didn't...,40 years old getting into farming didn't know...
5,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,daughter who came out of it actually she desig...,"daughter who came out of it, actually, she de..."
6,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,has her own business in that respect so there'...,has our own business in that respect. So ther...
7,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,advocating advocating and working on developin...,advocating and working on developing that. Th...
8,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,five point plan and you the folks that are run...,five point planning to have you the folks tha...
9,07282016HFUUforum_SLASH_07-28-2016_HFUUforum_D...,but domestic violence doesn't happen in a vacu...,But domestic violence doesn't happen in a vac...


In [ ]:
# Define the evaluation function
def calculate_wer(text: str, generated_text: str) -> float:
    # Safety check if ground-truth is empty
    if not text.strip():
        return 0.0 if not generated_text.strip() else 1.0

    return jiwer.wer(text, generated_text)

# Apply the mapping to your text_dc chain
wer_chain = (
    text_chain
    .map(wer=calculate_wer)
    .persist()
)

# Show the side-by-side comparison and the resulting score
wer_chain.order_by("wer").select("text", "generated_text", "wer").show()

Preparing: 0 rows [00:00, ? rows/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Processed: 0 rows [00:00, ? rows/s]

,text,generated_text,wer
0,farmers but gardeners and foodies alike who va...,"farmers, but gardeners and foodies alike who ...",0.131579
1,advocating advocating and working on developin...,advocating and working on developing that. Th...,0.189189
2,at forty years old getting into farming didn't...,40 years old getting into farming didn't know...,0.196429
3,five point plan and you the folks that are run...,five point planning to have you the folks tha...,0.212766
4,i know on a very small scale what farmers expe...,I know on a very small scale what farmers exp...,0.214286
5,has her own business in that respect so there'...,has our own business in that respect. So ther...,0.243902
6,but domestic violence doesn't happen in a vacu...,But domestic violence doesn't happen in a vac...,0.261905
7,i wanted this to share a few things but i'm go...,"I wanted to share a few things, but I'm not g...",0.265306
8,daughter who came out of it actually she desig...,"daughter who came out of it, actually, she de...",0.266667
9,three and we produce last year we produce twen...,"And we produced last year we produced 21,000 ...",0.363636


In [ ]:
average_wer_chain = wer_chain.avg("wer")

print(f"The final Mean WER for this batch is: {average_wer_chain:.4f}")

The final Mean WER for this batch is: 0.2476


## Normalizing the generated text

In [ ]:
# Build a normalization pipeline
normalizer = jiwer.Compose([
    jiwer.ToLowerCase(),                             # "Hello World" -> "hello world"
    jiwer.ExpandCommonEnglishContractions(),         # "you're" -> "you are"
    jiwer.RemovePunctuation(),                       # "hello, world!" -> "hello world"
    jiwer.RemoveWhiteSpace(replace_by_space=True),   # "hello\tworld\n" -> "hello world "
    jiwer.RemoveMultipleSpaces(),                    # "hello   world" -> "hello world"
    jiwer.Strip(),                                   # "  hello world  " -> "hello world"
])

# Update the evaluation function
def calculate_normalized_wer(text: str, generated_text: str) -> float:
    if not text.strip():
        return 0.0 if not generated_text.strip() else 1.0

    # Apply the normalizer to both strings BEFORE comparing them
    clean_truth = normalizer(text)
    clean_generated = normalizer(generated_text)

    # safety check
    if not clean_truth.strip():
        return 0.0 if not clean_generated.strip() else 1.0

    return jiwer.wer(clean_truth, clean_generated)

# Apply it to the chain
normalized_wer_chain = (
    text_chain
    .map(normalized_wer=calculate_normalized_wer)
    .persist()
)

normalized_wer_chain.order_by("normalized_wer").select("text", "generated_text", "normalized_wer").show()

UDF 'calculate_normalized_wer': Skipped, reusing output from checkpoint


,text,generated_text,normalized_wer
0,i know on a very small scale what farmers expe...,I know on a very small scale what farmers exp...,0.022727
1,advocating advocating and working on developin...,advocating and working on developing that. Th...,0.052632
2,at forty years old getting into farming didn't...,40 years old getting into farming didn't know...,0.065574
3,farmers but gardeners and foodies alike who va...,"farmers, but gardeners and foodies alike who ...",0.073171
4,i wanted this to share a few things but i'm go...,"I wanted to share a few things, but I'm not g...",0.078431
5,daughter who came out of it actually she desig...,"daughter who came out of it, actually, she de...",0.086957
6,but domestic violence doesn't happen in a vacu...,But domestic violence doesn't happen in a vac...,0.090909
7,has her own business in that respect so there'...,has our own business in that respect. So ther...,0.095238
8,five point plan and you the folks that are run...,five point planning to have you the folks tha...,0.122449
9,three and we produce last year we produce twen...,"And we produced last year we produced 21,000 ...",0.242424


In [ ]:
average_normalized_wer_chain = normalized_wer_chain.avg("normalized_wer")
print(f"The final Mean WER for this batch is: {average_normalized_wer_chain:.4f}")

The final Mean WER for this batch is: 0.1129


## Use whisper's normalizer

In [ ]:
# Download the spelling dictionary from the Whisper repo
mapping_file = hf_hub_download(
    repo_id="openai/whisper-tiny",
    filename="normalizer.json"
)

# Load it into a standard Python dictionary
with open(mapping_file, "r", encoding="utf-8") as f:
    english_spelling_mapping = json.load(f)

# Initialize the normalizer with the mapping
whisper_normalizer = EnglishTextNormalizer(english_spelling_mapping)

# Update the function to return a tuple of (str, str, float)
def normalize_and_evaluate(text: str, generated_text: str) -> tuple[str, str, float]:
    if not text.strip():
        return "", "", 0.0 if not generated_text.strip() else 1.0

    # Apply Whisper's normalizer
    clean_truth = whisper_normalizer(text)
    clean_generated = whisper_normalizer(generated_text)

    # Edge case: normalizer strips everything away
    if not clean_truth.strip():
        return clean_truth, clean_generated, 0.0 if not clean_generated.strip() else 1.0

    # Calculate the WER
    wer_score = jiwer.wer(clean_truth, clean_generated)

    return clean_truth, clean_generated, float(wer_score)

# Apply it to your chain using explicit params and outputs
whisper_wer_chain = (
    text_chain
    .map(
        func=normalize_and_evaluate,
        params=["text", "generated_text"],
        output={
            "normalized_text": str,
            "normalized_generated_text": str,
            "whisper_wer": float
        }
    )
    .persist()
)

# View the full side-by-side comparison!
whisper_wer_chain.order_by("whisper_wer").select(
    "text", "normalized_text",
    "generated_text", "normalized_generated_text",
    "whisper_wer"
).show()

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Processed: 0 rows [00:00, ? rows/s]

,text,normalized_text,generated_text,normalized_generated_text,whisper_wer
0,i know on a very small scale what farmers expe...,i know on a very small scale what farmers expe...,I know on a very small scale what farmers exp...,i know on a very small scale what farmers expe...,0.022727
1,at forty years old getting into farming didn't...,at 40 years old getting into farming did not k...,40 years old getting into farming didn't know...,40 years old getting into farming did not know...,0.04918
2,advocating advocating and working on developin...,advocating advocating and working on developin...,advocating and working on developing that. Th...,advocating and working on developing that 3rd ...,0.052632
3,commons here in the spirit of being able to gr...,commons here in the spirit of being able to gr...,comments here in the spirit of being able to ...,comments here in the spirit of being able to g...,0.055556
4,farmers but gardeners and foodies alike who va...,farmers but gardeners and foodies alike who va...,"farmers, but gardeners and foodies alike who ...",farmers but gardeners and foodies alike who va...,0.073171
5,i wanted this to share a few things but i'm go...,i wanted this to share a few things but i am g...,"I wanted to share a few things, but I'm not g...",i wanted to share a few things but i am not go...,0.078431
6,daughter who came out of it actually she desig...,daughter who came out of it actually she desig...,"daughter who came out of it, actually, she de...",daughter who came out of it actually she desig...,0.086957
7,but domestic violence doesn't happen in a vacu...,but domestic violence does not happen in a vac...,But domestic violence doesn't happen in a vac...,but domestic violence does not happen in a vac...,0.090909
8,has her own business in that respect so there'...,has her own business in that respect so there ...,has our own business in that respect. So ther...,has our own business in that respect so there ...,0.095238
9,three and we produce last year we produce twen...,3 and we produce last year we produce £21000 o...,"And we produced last year we produced 21,000 ...",and we produced last year we produced £21000 o...,0.103448


In [ ]:
# 1. Use the built-in .avg() method on your numeric column
average_whisper_wer_chain = whisper_wer_chain.avg("whisper_wer")

print(f"The final Mean WER for this batch is: {average_whisper_wer_chain:.4f}")

The final Mean WER for this batch is: 0.0755


In [ ]:
ordered_chain = whisper_wer_chain.order_by("whisper_wer").select(
    "text", "normalized_text",
    "generated_text", "normalized_generated_text",
    "whisper_wer"
)

# Convert the DataChain dataset to a pandas DataFrame
df = ordered_chain.to_pandas()

# Access the second row (index 1) as a dictionary
second_row_dict = df.iloc[9].to_dict()
second_row_dict


{'text': 'three and we produce last year we produce twenty one thousand pounds of food for the community on two thousand square feet so unless we were that efficient to produce that much food ',
 'normalized_text': '3 and we produce last year we produce £21000 of food for the community on 2000 square feet so unless we were that efficient to produce that much food',
 'generated_text': ' And we produced last year we produced 21,000 pounds of food for the community on 2,000 square feet. So unless we were that efficient to produce that much food,',
 'normalized_generated_text': 'and we produced last year we produced £21000 of food for the community on 2000 square feet so unless we were that efficient to produce that much food',
 'whisper_wer': 0.10344827586206896}